# Shared Urban + metric depth — decoder-only pilot
Revision **PARKING-SHARED-DEPTH-PILOT-2026-10-27-r1**.

This is the first **ground-truth-only** depth training pilot, not knowledge distillation. Urban's pretrained backbone, neck, detection/corner weights **and BatchNorm statistics stay frozen**. Only the new 57,441-parameter decoder is trained.

**Budget:** 512 training images, 128 development images, 5 epochs, batch 8, AdamW LR 0.001, weight decay 0.0001. Loss is per-image mean Huber in raw metres (delta 0.2), then an equal mean over images. No class/range reweighting, augmentation or per-image scale fitting.

The input package uses six official mini_train scenes for training, two different mini_train scenes for development, and the earlier fixed64 from the two official mini_val scenes only for final evaluation. Checkpoint selection is development MAE only. Sparse camera-aligned LiDAR is not dense/metrology ground truth. **Pedestrian 2–4 m is absent from held-out data; the full parking requirement cannot pass on this package.**

No iPhone model is replaced. First stage inputs on a CPU runtime, then switch Colab to GPU. No custom CUDA extension, Core ML or old M67 runtime is needed. The model takes RGB only; recorded camera intrinsics align sparse labels but are not an input to its depth head. nuScenes-to-iPhone camera/domain generalization and optical-axis depth versus target-surface/ultrasonic range remain unvalidated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, subprocess, shlex, datetime
REVISION = 'PARKING-SHARED-DEPTH-PILOT-2026-10-27-r1'
RUN_ID = 'shared_depth_gt_512_r1'
MOBILE = Path('/content/mobile_adas3d')
ARCHIVE = Path('/content/drive/MyDrive/mobile_adas3d_inputs/parking_shared_depth/inputs_r1/shared_depth_inputs.zip')
INPUTS = Path('/content/shared_depth_inputs_r1')
CACHE = Path('/content/shared_depth_features_' + RUN_ID)
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/parking_shared_depth') / RUN_ID
SOURCE = Path('/content/Urban_Shared_Depth_Source')
WEIGHTS = Path('/content/urban_shared_depth_weights')
DEPS = Path('/content/shared_depth_pilot_deps')
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(DEPS) + os.pathsep + str(MOBILE)
def run(command, name):
    command = [str(x) for x in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = OUTPUT / 'colab_logs' / (name + '_' + stamp + '.log')
    log.parent.mkdir(parents=True, exist_ok=True)
    print('+', shlex.join(command), '\nDurable log:', log, flush=True)
    tail = []
    with log.open('w') as stream:
        process = subprocess.Popen(command, cwd=MOBILE if MOBILE.exists() else '/content',
                                   env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end='', flush=True); stream.write(line); stream.flush()
            tail = (tail + [line.rstrip()])[-30:]
        process.stdout.close(); status = process.wait()
        stream.write('\nExit code: ' + str(status) + '\n')
    if status: raise RuntimeError(f'Exit {status}; preserve outputs and inspect {log}\n' + '\n'.join(tail))
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists(): run(['git', 'clone', url, MOBILE], 'clone')
branch = subprocess.check_output(['git','branch','--show-current'], cwd=MOBILE, text=True).strip()
if branch != 'main': raise RuntimeError('Use main; preserve local branches rather than resetting them')
run(['git','pull','--ff-only'], 'update')
for name in ['parking_shared_depth_data.py','train_parking_shared_depth.py']:
    if not (MOBILE/'scripts'/name).is_file(): raise FileNotFoundError('Training code not present on main: ' + name)
print(REVISION, '— CPU staging first')

## 2. Restore one reusable input archive — CPU only
Upload the prepared **shared_depth_inputs.zip** to the ARCHIVE location above. It contains the selected original images, sparse GT and a hash-protected split manifest. It does not contain model predictions.

Preparation on the Mac, if needed:
`PYTHONPATH=<nuScenes SDK dependency directory> .venv/bin/python scripts/parking_shared_depth_data.py prepare --dataset <nuScenes mini root> --test-inputs outputs/parking_lidar_depth_20261027/nuscenes_mini_val_r1 --output <new input-cache folder>`

Four workers stage/verify the archive once. Existing valid local files are reused. Do not copy hundreds of individual files from Drive inside the training loop.

In [ ]:
if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Upload the prepared training archive here first: {ARCHIVE}')
import hashlib
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
if digest(ARCHIVE) != '309e98e2eaca564817a3bd7e4ae0b751a112983c8dd7e042d81fec66d4f019b5':
    raise RuntimeError('Different input archive; preserve it and review a separate data revision')
run([sys.executable, MOBILE/'scripts/parking_shared_depth_data.py', 'restore',
     '--archive', ARCHIVE, '--output', INPUTS], 'stage_inputs_cpu')
assert digest(INPUTS/'input_manifest.json') == '31eadf50a5fe08baea0ef66e69849bd8c56b114f8e2d36e6350e7dceda897013'
manifest = json.loads((INPUTS/'input_manifest.json').read_text())
assert manifest['revision'] == REVISION
print(json.dumps(manifest['coverage'], indent=2))
print('Staging complete. Now select a GPU runtime in Colab; rerun section 1 after reconnecting.')

## 3. GPU preflight and tests
Run after selecting a GPU. Small pinned Urban Python packages are placed in a task-only directory; Colab's installed PyTorch/CUDA are recorded, not globally replaced. If Colab reconnects to a new machine, rerun sections 1–3 to restore local files and dependencies.

Completed epoch checkpoints are resumable with the same run identity. Interrupted work inside an epoch restarts from the previous completed epoch. Changed recipes/software/compute identities require a new RUN_ID; never edit an existing manifest to force a resume.

In [ ]:
run([sys.executable,'-m','pip','install','-q','--target',DEPS,'--no-deps',
     'ultralytics==8.4.61','ultralytics-thop==2.0.18'], 'task_dependencies')
run([sys.executable,'-c',
     "import torch,ultralytics; assert torch.cuda.is_available(), 'Choose a GPU runtime'; "
     "assert ultralytics.__version__=='8.4.61'; print(torch.__version__, torch.cuda.get_device_name(0))"],
    'cuda_preflight')
run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_parking_shared_depth_training.py','-v'],
    'training_tests')
SCRIPT = MOBILE/'scripts/train_parking_shared_depth.py'
BASE = ['--inputs',INPUTS,'--cache',CACHE,'--source',SOURCE,'--weights',WEIGHTS,'--output',OUTPUT,'--device','cuda']
def stage(name): run([sys.executable,'-X','faulthandler','-u',SCRIPT,name,*BASE], name)
print('Ready. The phone app and existing runs remain unchanged.')

## 4. Cache frozen features once
This is GPU inference only, with Urban BatchNorm in evaluation mode. It prepares training/development features but does not evaluate held-out images. The feature cache is local, hash-checked and FP32. Keep the local cache for the remaining stages.

In [ ]:
stage('cache')

## 5. Train only the new depth decoder — five epochs
No external teacher is used. All depth supervision is sparse metric ground truth. Each completed epoch verifies frozen Urban weights/BN and an exactly preserved detection probe, then saves decoder and optimizer state.

The lowest **development macro-image MAE** chooses the checkpoint. The fixed held-out set is not used for tuning or selection.

In [ ]:
stage('train')
selection = json.loads((OUTPUT/'selection.json').read_text())
assert selection['complete'] and selection['frozen_body_and_batchnorm_unchanged']
assert selection['test_used_for_selection'] is False
print('Selected epoch:', selection['selected_epoch'], 'Optimizer steps:', selection['optimizer_steps'])
for row in selection['history']:
    print('Epoch',row['epoch'],'dev MAE (m):',round(row['dev']['macro_image_mae_m'],4))

## 6. Open held-out data once, score raw metres, and bundle
Compare trained depth against the untrained initialization on exactly the same sparse reference points. Report Vehicle/Pedestrian 2–4, 4–6 and 2–6 m, signed/absolute errors and fractions within 10/20 cm. Missing cells remain missing.

This does not measure detection mAP, calibrated cuboid quality, independent iPhone accuracy, or ultrasound sensor range. Do not choose another checkpoint based on this held-out result. Return the result zip for review before further changes.

In [ ]:
stage('evaluate')
run([sys.executable,SCRIPT,'bundle','--output',OUTPUT], 'bundle')
result = json.loads((OUTPUT/'depth_pilot_results.json').read_text())
assert result['complete'] and result['frozen_urban_weights_and_bn_unchanged']
print('Selected epoch:',result['selected_epoch'])
for key in ['conservative/Vehicle/2-6','conservative/Pedestrian/2-6','conservative/Pedestrian/2-4']:
    print(key, json.dumps(result['trained_decoder']['cells'][key], indent=2))
print('Qualified for deployment:', result['deployment_qualified'])
print('Return:', OUTPUT/'parking_shared_depth_results.zip')
from google.colab import files
files.download(str(OUTPUT/'parking_shared_depth_results.zip'))